# Hisse Tahmini (Sequence NN)

Bu projede AAPL kapanışını gecikmeli özelliklerle tahmin edeceğim. LSTM yerine MLPRegressor kullandım.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt


### Data


In [ ]:
df=pd.read_csv('data/AAPL.csv',parse_dates=['Date']).sort_values('Date')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


### Görselleştirme


In [ ]:
plt.plot(df['Date'],df['Close'])
plt.show()


### Boş veri


In [ ]:
df['Close']=df['Close'].ffill()


### Feature Engineering


In [ ]:
s=df.copy()
for i in range(1,6):
    s[f'lag{i}']=s['Close'].shift(i)
s=s.dropna()
x=s[[f'lag{i}' for i in range(1,6)]]
y=s['Close']


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,shuffle=False)


### 3 Model


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error

for ad,m in [('MLP',MLPRegressor(hidden_layer_sizes=(32,16),max_iter=400,random_state=42)),('LR',LinearRegression()),('RF',RandomForestRegressor(random_state=42))]:
    m.fit(x_train,y_train)
    p=m.predict(x_test)
    print(ad,round(r2_score(y_test,p),3),round(mean_absolute_error(y_test,p),3))


### Feature Importance + Residual


In [ ]:
rf=RandomForestRegressor(random_state=42).fit(x_train,y_train)
print(pd.Series(rf.feature_importances_,index=x.columns))
pred=rf.predict(x_test)
plt.scatter(pred,y_test-pred)
plt.axhline(0,color='r')
plt.show()


In [ ]:
import joblib
joblib.dump(rf,'../../models/dl_stock_lstm.joblib')


### Sonuç

Lag1 yine dominant. MLP burada linear kadar iyi değil. Hedefi kısmen tutturdum.
